1. ChatMessageHistory 的使用

- 场景1：记忆存储

In [1]:
from langchain.memory import ChatMessageHistory

# 1. ChatMessageHistory 的实例化
history = ChatMessageHistory()

# 2. 添加相关的消息进行存储
history.add_user_message("你好")

history.add_ai_message("你好，很高兴认识你")

# 3. 打印存储的消息
print(history.messages)

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，很高兴认识你', additional_kwargs={}, response_metadata={})]


- 场景2：对接大模型

In [2]:
# 获取大模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain.memory import ChatMessageHistory

# 1. ChatMessageHistory 的实例化
history = ChatMessageHistory()

# 2. 添加相关的消息进行存储
history.add_user_message("你好")
history.add_ai_message("你好，很高兴认识你")
history.add_user_message("帮我计算1 + 2 * 3 = ?")

response = llm.invoke(history.messages)
print(response.content)


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


根据运算顺序（先乘法后加法），计算步骤如下：

1. 先计算乘法：2 * 3 = 6
2. 然后再进行加法：1 + 6 = 7

所以，1 + 2 * 3 = 7。


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2. ConversationBufferMemory的使用

- 举例1: 以字符串的方式返回存储的信息

In [ ]:
from langchain.memory import ConversationBufferMemory

# 1. ConversationBufferMemory的实例化
memory = ConversationBufferMemory()

# 2. 存储相关的消息
# inputs对应的就是用户消息, outputs对应的就是ai的消息
memory.save_context(inputs={"human": '你好，我叫小明'}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": '帮我回答一下 1 + 2 * 3 = ?'}, outputs={"output": "7"})

# 3. 获取存储的信息
print(memory.load_memory_variables({}))

# 说明：返回的字典结构的key叫history


{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我回答一下 1 + 2 * 3 = ?\nAI: 7'}


- 举例2: 以消息列表的方式返回存储的信息

In [11]:
from langchain.memory import ConversationBufferMemory

# 1. ConversationBufferMemory的实例化
memory = ConversationBufferMemory(return_messages=True)

# 2. 存储相关的消息
# inputs对应的就是用户消息, outputs对应的就是ai的消息
memory.save_context(inputs={"human": '你好，我叫小明'}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": '帮我回答一下 1 + 2 * 3 = ?'}, outputs={"output": "7"})

# 3. 获取存储的信息
# 返回消息列表的方式一
print(memory.load_memory_variables({}))

print("\n")

# 返回消息列表的方式二
print(memory.chat_memory.messages)

# 说明：返回的字典结构的key叫history

{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下 1 + 2 * 3 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={})]}


[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下 1 + 2 * 3 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={})]


- 举例3：结合大模型、提示词模板的使用（PromptTemplate）

In [12]:
# 获取大模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [14]:
# 提供提示词模板
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人对话

    当前对话的历史：{history}
    人类回答：{question}
    回复：
    """
)

# 提示memory实例
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory()

# 提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
response = chain.invoke({"question": "你好，我的名字叫小明"})
print(response)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_24992\2538651560.py:20: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'question': '你好，我的名字叫小明', 'history': '', 'text': '你好，小明！很高兴认识你。你今天过得怎么样？'}


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [15]:
response = chain.invoke({"question": "我叫什么名字呢？"})
print(response)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'question': '我叫什么名字呢？', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你。你今天过得怎么样？', 'text': '你叫小明。很高兴再次和你交流！你有什么想聊的呢？'}


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


- 举例4：基于举例3，显式的设置memory的key的值

In [ ]:
# 提供提示词模板
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人对话

    当前对话的历史：{chat_history}
    人类回答：{question}
    回复：
    """
)

# 提示memory实例
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history")

# 提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
response = chain.invoke({"question": "你好，我的名字叫小明"})
print(response)

- 举例5：结合大模型、提示词模板的使用（ChatPromptTemplate）

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import LLMChain

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个与人对话的机器人"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "问题：{question}"),
])

# 提示memory实例
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(return_messages=True)

# 提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
resp1 = chain.invoke({"question": "中国的首都在哪里？"})
print(resp1, end="\n\n")

resp2 = chain.invoke({"question": "我刚刚问了什么？"})
print(resp2)


{'question': '中国的首都在哪里？', 'history': [HumanMessage(content='中国的首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都位于北京。', additional_kwargs={}, response_metadata={})], 'text': '中国的首都位于北京。'}



Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'question': '我刚刚问了什么？', 'history': [HumanMessage(content='中国的首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都位于北京。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我刚刚问了什么？', additional_kwargs={}, response_metadata={}), AIMessage(content='您问的是“中国的首都在哪里？”', additional_kwargs={}, response_metadata={})], 'text': '您问的是“中国的首都在哪里？”'}


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


3. ConversationChain的使用

- 举例1：以PromptTemplate为例

In [22]:
# 提供提示词模板
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain, ConversationChain

prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人对话

    当前对话的历史：{history}
    人类回答：{input}
    回复：
    """
)

# 提示memory实例
# from langchain.memory import ConversationBufferMemory
# memory = ConversationBufferMemory()

# 提供Chain
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
# response = chain.invoke({"question": "你好，我的名字叫小明"})

# 创建ConversationChain的实例
chain = ConversationChain(llm=llm, prompt=prompt_template)
response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'input': '你好，我的名字叫小明', 'history': '', 'response': '你好，小明！很高兴认识你。你今天过得怎么样？'}


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


- 举例2：使用默认提供的提示词模板

In [ ]:
# 提供提示词模板
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain, ConversationChain

# prompt_template = PromptTemplate.from_template(
#     template="""
#     你可以与人对话

#     当前对话的历史：{history}
#     人类回答：{input}
#     回复：
#     """
# )

# 提示memory实例
# from langchain.memory import ConversationBufferMemory
# memory = ConversationBufferMemory()

# 提供Chain
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
# response = chain.invoke({"question": "你好，我的名字叫小明"})

# 创建ConversationChain的实例
chain = ConversationChain(llm=llm)
response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)